# 📝 Day 2: Building Agents with Google ADK

**: 10 **

| | | |
|---------|-------|--------|
| 1. Agent + Custom Tool | 3 | Agent tool |
| 2. RAG Agent | 3 | Agent Qdrant |
| 3. Workflow Agent | 2 | Sequential/Parallel/Loop pipeline |
| 4. | 2 | |

### ⚠️ 
- **** → 
- **** cell 
- **Run cell** 

## 📦 Dependencies

In [ ]:
%%time
!pip install -q google-adk google-genai sentence-transformers qdrant-client langchain-text-splitters

import hashlib, os, json, random
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings('ignore')

print('✅ !')

In [ ]:
%%time
# Gemini API
try:
 from google.colab import userdata
 os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
 print('✅ API Key Colab Secrets')
except Exception:
 api_key = input('🔑 Gemini API Key: ')
 os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'
print('✅ API Key ')

## 🎓 

In [ ]:
# ─── ───
STUDENT_NAME = '' # ' '
STUDENT_ID = '' # '6512345678'
PHONE = '' # '081-234-5678'
LINE_ID = '' # 'somchai.j'

# ─── () ───
assert len(STUDENT_ID) >= 5, '❌ !'
assert len(STUDENT_NAME) >= 2, '❌ -!'

print(f'✅ -: {STUDENT_NAME}')
print(f'✅ : {STUDENT_ID}')
print(f'📱 : {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 ()

> → 

In [ ]:
# ===== cell =====
random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_topics = [
 {'topic': 'healthcare', 'text': ' AI X-ray 30 5 Deep Learning 95% '},
 {'topic': 'banking', 'text': ' KADE AI Assistant RAG 5 30 '},
 {'topic': 'education', 'text': ' RAG - PDF 500 vector embeddings multilingual model '},
 {'topic': 'agriculture', 'text': ' AI 92% 40%'},
 {'topic': 'logistics', 'text': 'Kerry Express AI 15% 200 Machine Learning '},
 {'topic': 'retail', 'text': ' AI Recommendation System 25% 10 '},
 {'topic': 'energy', 'text': ' (EGAT) AI 8% Deep Learning Smart Meter '},
 {'topic': 'tourism', 'text': '. AI Chatbot 5 1 call center 60%'},
 {'topic': 'insurance', 'text': ' AI 7 1 fraud 90%'},
 {'topic': 'manufacturing', 'text': 'SCG AI Computer Vision 30% Predictive Maintenance 2 '},
]

random.shuffle(all_topics)
my_topics = all_topics[:6] # 6 10 

# Qdrant
embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=30)

my_chunks = []
my_sources = []
for item in my_topics:
 chunks = splitter.split_text(item['text'])
 for c in chunks:
 my_chunks.append(c)
 my_sources.append(item['topic'])

# Embed + Index
passages = [f'passage: {c}' for c in my_chunks]
embeddings = embed_model.encode(passages)
VECTOR_SIZE = embeddings.shape[1]

qdrant = QdrantClient(':memory:')
collection_name = f'hw2_{STUDENT_ID}'
qdrant.create_collection(
 collection_name=collection_name,
 vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

points = [
 models.PointStruct(id=i, vector=embeddings[i].tolist(),
 payload={'text': my_chunks[i], 'source': my_sources[i]})
 for i in range(len(my_chunks))
]
qdrant.upsert(collection_name=collection_name, points=points)

# query 
query_pool = [
 'AI ',
 '',
 ' Machine Learning',
 'Chatbot ',
 'Computer Vision ',
 ' Deep Learning',
]
random.shuffle(query_pool)
MY_QUERY = query_pool[0]

# tool type
tool_types = [
 {'name': ' ROI AI', 'func': 'calculate_ai_roi', 'params': 'investment_baht: float, savings_per_year: float, years: int'},
 {'name': ' AI', 'func': 'assess_ai_readiness', 'params': 'num_employees: int, has_data: bool, budget_million: float'},
 {'name': ' Accuracy Score', 'func': 'calculate_accuracy', 'params': 'true_positives: int, false_positives: int, false_negatives: int, true_negatives: int'},
 {'name': ' AI Model', 'func': 'recommend_model', 'params': 'task_type: str, data_size: str, budget: str'},
]
random.shuffle(tool_types)
MY_TOOL = tool_types[0]

# Workflow type
workflow_types = ['SequentialAgent', 'ParallelAgent', 'LoopAgent']
random.shuffle(workflow_types)
MY_WORKFLOW = workflow_types[0]

print(f'✅ !')
print(f' 📊 : {[t["topic"] for t in my_topics]}')
print(f' 📦 Chunks: {len(my_chunks)} | Collection: {collection_name}')
print(f' 🔍 Query : "{MY_QUERY}"')
print(f' 🔧 Tool : {MY_TOOL["name"]}')
print(f' 🔄 Workflow : {MY_WORKFLOW}')

## 🔧 ()

In [ ]:
# ===== cell =====
async def chat_with_agent(agent, message, user_id='user_1', session_id=None):
 """ Agent """
 if session_id is None:
 session_id = f'session_{id(agent)}'
 runner = InMemoryRunner(agent=agent, app_name=agent.name)
 session = await runner.session_service.create_session(
 app_name=agent.name, user_id=user_id, session_id=session_id)
 content = types.Content(role='user', parts=[types.Part.from_text(text=message)])

 final_response = ''
 tool_calls = []
 async for event in runner.run_async(
 user_id=user_id, session_id=session.id, new_message=content):
 if event.content and event.content.parts:
 for part in event.content.parts:
 if part.text: final_response = part.text
 if part.function_call:
 tool_calls.append(part.function_call.name)

 if tool_calls:
 print(f' 🔧 Tools: {tool_calls}')
 return final_response, session.id

print('✅ chat ')


---
## 🎯 1: Agent + Custom Tool (3 )

 Agent **2 tools**:
1. `search_knowledge` — Qdrant ()
2. **Custom Tool** — ( `MY_TOOL`)

### 
- 1 : custom tool ( docstring + type hints + return dict)
- 1 : Agent 2 tools 
- 1 : Agent tool 

In [ ]:
# RAG Tool ( — )
def search_knowledge(query: str) -> dict:
 """ AI 
 
 Args:
 query: 
 """
 q_vec = embed_model.encode(f'query: {query}').tolist()
 results = qdrant.query_points(
 collection_name=collection_name, query=q_vec, limit=3).points
 return {
 'status': 'success',
 'results': [{'text': r.payload['text'], 'source': r.payload['source'],
 'score': round(r.score, 4)} for r in results]
 }

print('✅ RAG Tool ')
print(f'\n📌 Tool : {MY_TOOL["name"]}')
print(f' : {MY_TOOL["func"]}({MY_TOOL["params"]})')
print(f' return dict')

In [ ]:
# 1: 

# 💡 Hint:
# 1. MY_TOOL 
# 2. docstring (LLM !)
# 3. type hints
# 4. return dict

# def MY_TOOL['func'](...) -> dict:
# """..."""
# ...
# return {...}



# Agent 2 tools
# my_agent = Agent(
# name='my_agent',
# model='gemini-2.5-flash',
# tools=[search_knowledge, my_custom_tool]
# )



# Tool Selection ( 3 )
# 1. RAG tool
# 2. custom tool
# 3. ( tool)
# await chat_with_agent(my_agent, '...')

---
## 🎯 2: RAG Agent — (3 )

 Agent Qdrant **query **

### 
- 1 : Agent + Top-3 scores
- 1 : 
- 1 : chunk (3-5 )

In [ ]:
# 2: 

print(f'🔍 Query : "{MY_QUERY}"')

# 💡 Hint:
# 1. Agent instruction search_knowledge
# 2. MY_QUERY
# 3. Top-3 scores
# 4. ( AI)

# rag_agent = Agent(
# name='rag_agent',
# model='gemini-2.5-flash',
# instruction='...',
# tools=[search_knowledge]
# )
# response, _ = await chat_with_agent(rag_agent, MY_QUERY)
# print(response)

In [ ]:
# 2 (): 
# 3-5 :
# 1. chunks ?
# 2. query ?
# 3. ?

MY_EXPLANATION_STEP2 = '''
()
'''
print(MY_EXPLANATION_STEP2)

---
## 🎯 3: Workflow Agent (2 )

 Workflow **pattern ** `MY_WORKFLOW`

### 
- 1 : Workflow Agent pattern
- 1 : + 

In [ ]:
# 3: 

print(f'🔄 Workflow : {MY_WORKFLOW}')

# 💡 Hint pattern:
#
# SequentialAgent:
# from google.adk.agents import SequentialAgent
# pipeline = SequentialAgent(sub_agents=[agent1, agent2, agent3])
# → → → ()
#
# ParallelAgent:
# from google.adk.agents import ParallelAgent
# parallel = ParallelAgent(sub_agents=[search1, search2, search3])
# → 
#
# LoopAgent:
# from google.adk.agents import LoopAgent
# loop = LoopAgent(sub_agents=[writer, critic], max_iterations=3)
# → ↔ 



# 
# response, _ = await chat_with_agent(my_workflow, MY_QUERY)
# print(response)

---
## 🎯 4: (2 )

### 
- 1 : Workflow — ? /?
- 1 : 2 patterns — pattern ?

In [ ]:
# 4: 

MY_EXPLANATION_STEP4 = f'''
Pattern : {MY_WORKFLOW}

1. {MY_WORKFLOW} :
 ()

2. {MY_WORKFLOW}:
 ()

3. {MY_WORKFLOW}:
 ()

4. pattern :
 ()
'''
print(MY_EXPLANATION_STEP4)

## ✅ 

In [ ]:
# ===== cell =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_day2_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 -: {STUDENT_NAME}')
print(f'🎓 : {STUDENT_ID}')
print(f'📱 : {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 : _________________ ()')
print('=' * 50)
print()
print(f'📌 :')
print(f' : {[t["topic"] for t in my_topics]}')
print(f' Query: {MY_QUERY}')
print(f' Tool: {MY_TOOL["name"]}')
print(f' Workflow: {MY_WORKFLOW}')
print()
print('📋 Checklist :')
print(' [ ] ')
print(' [ ] 1: Agent + Custom Tool + 3 ')
print(' [ ] 2: RAG Agent + Top-3 scores + 3-5 ')
print(' [ ] 3: Workflow Agent pattern ')
print(' [ ] 4: + ')
print(' [ ] cell run ')